In [1]:
import os
import json
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
from torch.nn.functional import one_hot
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
from torch.optim import AdamW
from torch.optim import lr_scheduler
from torch.amp import autocast
from torch.nn.utils.rnn import pad_sequence
from torchvision import transforms
from torchvision.transforms import v2
import torchvision

from Swin_classif import SwinTransformerClassification


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# Dataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 2. Install the Kaggle library
!pip install -q kaggle

# 3. Download the dataset directly
!kaggle datasets download -d ambityga/imagenet100

# 4. Unzip the downloaded file
!unzip -q imagenet100.zip -d imagenet100
!rm imagenet100.zip

Dataset URL: https://www.kaggle.com/datasets/ambityga/imagenet100
License(s): unknown
100% 16.1G/16.1G [14:01<00:00, 20.6MB/s]



In [2]:
labels = json.load(open("/content/imagenet100/Labels.json"))

# Load training data
images = []
for subset_dirname in os.listdir('/content/imagenet100'):
  if subset_dirname != "Labels.json" and subset_dirname != "val.X":
    for dirname in os.listdir(f'/content/imagenet100/{subset_dirname}'):
      for filename in os.listdir(f'/content/imagenet100/{subset_dirname}/{dirname}'):
        images.append({
            'path': f'/content/imagenet100/{subset_dirname}/{dirname}/{filename}',
            'label': labels[dirname],
        })

df_training = pd.DataFrame(images)
print(f"Total training images: {len(df_training)}")
print(f"top 3 training samples: {df_training[:3]}")

# Load validation data
images = []
for label in os.listdir(f'/content/imagenet100/val.X'):
    for filename in os.listdir(f'/content/imagenet100/val.X/{label}'):
        images.append({
            'path': f'/content/imagenet100/val.X/{label}/{filename}',
            'label': labels[label],
        })

df_val = pd.DataFrame(images)
print(f"Total val images: {len(df_val)}")
print(f"top 3 val samples: {df_val[:3]}")

Total training images: 130000
top 3 training samples:                                                 path  \
0  /content/imagenet100/train.X3/n01685808/n01685...   
1  /content/imagenet100/train.X3/n01685808/n01685...   
2  /content/imagenet100/train.X3/n01685808/n01685...   

                       label  
0  whiptail, whiptail lizard  
1  whiptail, whiptail lizard  
2  whiptail, whiptail lizard  
Total val images: 5000
top 3 val samples:                                                 path    label
0  /content/imagenet100/val.X/n02018795/ILSVRC201...  bustard
1  /content/imagenet100/val.X/n02018795/ILSVRC201...  bustard
2  /content/imagenet100/val.X/n02018795/ILSVRC201...  bustard


In [3]:
class Imagenet100Dataset(Dataset):
  def __init__(self, imagenet100_dataset, mapping_label_id, split, image_size=(224, 224)):
    super(Imagenet100Dataset, self).__init__()
    self.imagenet100_dataset = imagenet100_dataset
    self.mapping_label_id = mapping_label_id
    self.n_class = len(mapping_label_id)
    self.image_size = image_size

    # self.transform = transforms.Compose([
    #     transforms.Resize((512, 512)),
    #     transforms.ToTensor()
    # ])
    if split == 'train':
        self.transform = v2.Compose([
            v2.RandomResizedCrop(self.image_size, scale=(0.6, 1.0)),
            v2.RandomHorizontalFlip(p=0.5),
            v2.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    else:
        self.transform = v2.Compose([
            v2.Resize(self.image_size),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

  def __len__(self):
    return len(self.imagenet100_dataset)

  def __getitem__(self, idx):
    image_path = self.imagenet100_dataset.iloc[idx]['path']
    image = Image.open(image_path).convert("RGB")

    label = self.imagenet100_dataset.iloc[idx]['label']
    id = torch.tensor([self.mapping_label_id[label]], dtype=torch.long)

    return self.transform(image), id

In [4]:
def custom_collate_fn(batch):
    batch = [item for item in batch if item[0] is not None]
    if not batch:
        return None, None

    images, ids = zip(*batch)

    return torch.stack(images), torch.stack(ids)

In [5]:
# training data
torch.manual_seed(42)
batch_size = 256
mapping_label_id = {}
mapping_id_label = {}
for i, label in enumerate(labels.values()):
  mapping_label_id[label] = i
  mapping_id_label[i] = label

training_dataset = Imagenet100Dataset(df_training, mapping_label_id=mapping_label_id, split="train", image_size=(224,224))
validating_dataset = Imagenet100Dataset(df_val, mapping_label_id=mapping_label_id, split="val", image_size=(224,224))

training_dataloader = DataLoader(training_dataset, batch_size = batch_size,  collate_fn = custom_collate_fn, drop_last=True, num_workers=4)
validating_dataloader = DataLoader(validating_dataset, batch_size = batch_size,  collate_fn = custom_collate_fn, drop_last=False, num_workers=4)

# Model

In [6]:
model = SwinTransformerClassification(
    backbone_weights_path = None,
    backbone_input_size = (512, 512),
    backbone_patch_size = 4,
    backbone_patch_merging_ratio = 2,
    backbone_in_channels = 3,
    backbone_layers = [2, 2, 2, 2],
    backbone_query_size = 32,
    backbone_n_heads = [2, 4, 8, 16],
    backbone_mlp_factor=4,
    backbone_window_size = 7,
    n_classes = 100,
).to(device)

# TRAINING

In [14]:
def train_one_epoch(i,  model, device, loader, optimizer, scheduler, max_learning_rate, output_dir, accumulation_step=8):
    model.train()
    losses_displayed = []
    total_losses = []
    step = 0
    optimizer.zero_grad()

    for image_src, ids in loader:
        step += 1

        image_src = image_src.to(device)
        ids = ids.to(device)

        with autocast(device_type=device, dtype=torch.bfloat16):
            pred = model.forward(image_src)
            loss = torch.nn.functional.cross_entropy(pred, ids.squeeze())
            correct = (pred == ids).sum().item()
            accuracy = correct / len(ids)
            print(accuracy)

        loss = loss / accumulation_step
        loss.backward()
        if step%accumulation_step==0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            # we can set a maximum learning rate to avoid divergence
            for param_group in optimizer.param_groups:
                param_group['lr'] = min(param_group['lr'], max_learning_rate)


        losses_displayed.append(loss.item())
        total_losses.append(loss.item())

        if step%100 == 0:
            print(f"training loss mean on last 100 steps : {sum(losses_displayed)/len(losses_displayed)}")
            losses_displayed = []

    print(f"total training loss on the epoch : {sum(total_losses)/len(total_losses)}")
    FINE_TUNED_MODEL = output_dir + "/Swin_classif_epoch_"+ str(i)+".pt"
    torch.save(model.state_dict(), FINE_TUNED_MODEL)

    return model, optimizer, scheduler

In [15]:
def evaluate(model, device, loader):
    model.eval()
    losses_displayed = []
    total_losses = []
    step = 0
    optimizer.zero_grad()

    for image_src, ids in loader:
        step += 1

        image_src = image_src.to(device)
        ids = ids.to(device)

        with torch.no_grad():
          with autocast(device_type=device, dtype=torch.bfloat16):
              pred = model.forward(image_src)
          loss = torch.nn.functional.cross_entropy(pred, ids.squeeze())
          correct = (pred == ids).sum().item()
          accuracy = correct / len(ids)
          print(accuracy)

        losses_displayed.append(loss.item())
        total_losses.append(loss.item())

    print(f"total validation loss on the epoch : {str(sum(total_losses)/len(total_losses))}")
    return sum(total_losses) / len(total_losses)

## Warmup

In [9]:
n_epochs = 20
max_learning_rate = 0.0001

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=5e-4,
    steps_per_epoch=len(training_dataloader),
    epochs=n_epochs,
    pct_start=0.2,
    anneal_strategy='cos'
)

output_dir = "/content/drive/MyDrive/output_model"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
#model.load_state_dict(torch.load("/content/drive/MyDrive/output_model/Swin_tiny_Object_Detection_Pascal_VOC_epoch_30.pt"))

In [16]:
#track metrics
losses_val = []

model.train()
for i in range(n_epochs):
    print(f"Epoch {i+1}")
    model, optimizer, scheduler = train_one_epoch(i, model, str(device), training_dataloader, optimizer, scheduler, max_learning_rate, output_dir, accumulation_step=1)
    print(f"Evaluation ")
    loss = evaluate(model, str(device), validating_dataloader)
    losses_val.append(loss)


Epoch 1
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.01171875
0.03125
0.046875
0.03515625
0.04296875
0.0703125
0.05078125
0.03125
0.0234375
0.03125
0.015625
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
training loss mean on last 100 steps : 3.051247978210449
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.00390625
0.10546875
0.078125
0.1015625
0.1171875
0.0859375
0.00390625
0.0
training loss mean on last 10

KeyboardInterrupt: 

# Debug

In [ ]:
def show_image(image, bounding_boxes, labels, mapping_id_class):
    # 1. Handle PyTorch Image Tensor format (C, H, W) -> (H, W, C)
    if isinstance(image, torch.Tensor):
        # Move to CPU, detach from graph, and reorder dimensions for Matplotlib
        image = image.detach().cpu().permute(1, 2, 0).numpy()

        # If your image pixels are in range [0, 1], Matplotlib handles it fine.
        # If they are normalized with ImageNet mean/std, they might look washed out,
        # but the boxes will still align perfectly!

    # Convert boxes and labels to CPU numpy arrays if they are tensors
    if isinstance(bounding_boxes, torch.Tensor):
        bounding_boxes = bounding_boxes.detach().cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.detach().cpu().numpy()

    print(f"Image shape: {image.shape}, Boxes shape: {bounding_boxes.shape}, Labels shape: {labels.shape}")

    # Create a figure and axis
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image)

    # Draw bounding boxes
    for box, label in zip(bounding_boxes, labels):
        # 2. Pascal VOC standard format is [xmin, ymin, xmax, ymax]
        ymin, xmin, height, width = box

        # Get the string class name
        class_name = mapping_id_class.get(int(label), f"Class {int(label)}")

        # Create a rectangle patch
        rect = patches.Rectangle((xmin, ymin), width, height,
                                 linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)

        # 3. Add text label slightly above the bounding box
        ax.text(xmin, ymin - 5, class_name, color='white', fontsize=12, fontweight='bold',
                bbox=dict(facecolor='red', alpha=0.5, edgecolor='none', pad=2))

    # Hide axis and show image
    ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
def denormalize_and_convert_to_pil(tensor):

    img = tensor.clone()
    # img = img * std + mean  # Undo normalization -> back to [0,1]
    img = img.clamp(0, 1)  # Ensure range is valid
    img = img.mul(255).byte()  # Scale to [0,255] and convert to uint8
    img = img.permute(1, 2, 0)  # Convert from CxHxW to HxWxC for PIL
    return img.cpu().numpy()

In [ ]:
indice = 0
# Simulate data from the dataloader
filename, image, bounding_boxes, labels = next(iter(testing_dataloader))


# Convert image tensor to numpy and scale to [0, 255]
image = (image[indice].permute(1, 2, 0).numpy() * 255).astype(np.uint8)

# Apply mask to remove invalid bounding boxes
mask_padding = labels[indice] != -1
bounding_boxes = bounding_boxes[indice][mask_padding]
labels = labels[indice][mask_padding]
show_image(image, bounding_boxes, labels, mapping_id_class)


In [ ]:
indice = 1
model.eval()
# Simulate data from the dataloader
image, bounding_boxes, labels = next(iter(training_dataloader))

confidence_score, ROIs, pred_boxes, labels = model.forward(image)

print(confidence_score.shape)
print([ROI.shape for ROI in ROIs])
print([pred_boxe.shape for pred_boxe in pred_boxes])
print([label.shape for label in labels])

In [ ]:
indice = 5
model.eval()
# Simulate data from the dataloader
filename, image, bounding_boxes, labels = next(iter(training_dataloader))
image = image.to(device)
bounding_boxes = bounding_boxes.to(device)
labels = labels.to(device)
with torch.no_grad():
  confidence_score, ROIs, pred_boxes, labels = model.forward(image)

# Convert image tensor to numpy and scale to [0, 255]
image = (image[indice].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)

# Apply mask to remove invalid bounding boxes
mask_padding = labels[indice] != -1
pred_box = pred_boxes[indice][mask_padding]
print(pred_box.shape)
label = labels[indice][mask_padding]
show_image(image, pred_box, label, mapping_id_class)


In [ ]:
for name, param in model.backbone.named_parameters():
    param.requires_grad = False

In [ ]:
for name, param in model.OD_head.detector.named_parameters():
    param.requires_grad = False

In [ ]:
for name, param in model.named_parameters():
    print(name, param.requires_grad)